# EXP01 — Row-wise Dual Regression

Eksperimen ini membangun baseline **row-wise dual regression** untuk memprediksi `team_goals` dan `opp_goals`.

Karakter eksperimen:
- setiap baris dianggap satu observasi,
- melatih **2 model terpisah**,
- validasi wajib **time-based** dan **anti-leakage** per `match_id`,
- evaluasi utama menggunakan **official offline AW-MAE** pada **level pertandingan**,
- hasil akhir disimpan ke **`exp01_rowwise_dual_regression.csv`**.

Catatan penting:
- `ground_truth_bersih.csv` **tidak digunakan** untuk training, tuning, feature engineering, calibration, evaluasi, maupun inference,
- eksperimen baseline ini **tidak menggunakan rolling tambahan**.

## 1. Setup

Section ini menyiapkan import, seed, dan konfigurasi dasar supaya notebook runnable dari atas ke bawah.

In [ ]:
import os
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

SEED = 42

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
EXPERIMENT_NAME = "exp01_rowwise_dual_regression"
FINAL_CSV_NAME = f"{EXPERIMENT_NAME}.csv"

## 2. Load Data

Dataset yang dipakai:
- `train.csv` sebagai data utama
- `test.csv` sebagai data inference final

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)

display(train.head())
display(test.head())

## 3. EDA Singkat

EDA minimal yang ditampilkan:
- shape data
- daftar kolom
- missing values
- distribusi target
- jumlah `match_id` unik

In [ ]:
print("Daftar kolom train:")
display(pd.DataFrame({"column": train.columns}))

missing_summary = (
    train.isna().sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing_summary[missing_summary["missing_count"] > 0].head(30))

print("Jumlah match unik train:", train["match_id"].nunique())
print("Jumlah match unik test :", test["match_id"].nunique())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train["team_goals"].hist(ax=axes[0], bins=20)
axes[0].set_title("Distribusi team_goals")
train["opp_goals"].hist(ax=axes[1], bins=20)
axes[1].set_title("Distribusi opp_goals")
plt.tight_layout()
plt.show()

## 4. Validation Strategy

Validasi harus memenuhi aturan berikut:
- berbasis waktu (`date`)
- satu `match_id` tidak boleh pecah antara train dan validation
- evaluasi resmi dihitung **sekali per pertandingan**

Untuk baseline ini, dipakai holdout:
- 90% pertandingan paling awal untuk train
- 10% pertandingan paling akhir untuk validation

In [ ]:
def add_date_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date_year"] = df["date"].dt.year
    df["date_month"] = df["date"].dt.month
    df["date_day"] = df["date"].dt.day
    df["date_dayofweek"] = df["date"].dt.dayofweek
    df["date_dayofyear"] = df["date"].dt.dayofyear
    df["date_quarter"] = df["date"].dt.quarter
    df["date_is_month_start"] = df["date"].dt.is_month_start.astype(int)
    df["date_is_month_end"] = df["date"].dt.is_month_end.astype(int)
    df["date_ordinal"] = (df["date"] - pd.Timestamp("1970-01-01")).dt.days
    return df


def build_feature_frame(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr = add_date_features(train_df)
    te = add_date_features(test_df)

    feature_cols = [c for c in te.columns if c not in ["Id", "match_id", "date"]]
    combined = pd.concat([tr[feature_cols], te[feature_cols]], axis=0, ignore_index=True)

    num_cols = combined.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]

    for c in num_cols:
        if tr[c].isna().any() or te[c].isna().any():
            tr[f"{c}_missing"] = tr[c].isna().astype(int)
            te[f"{c}_missing"] = te[c].isna().astype(int)

    feature_cols = [c for c in tr.columns if c in feature_cols or c.endswith("_missing")]
    cat_cols = [c for c in feature_cols if c in cat_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    for c in cat_cols:
        tr[c] = tr[c].astype(str).fillna("__MISSING__")
        te[c] = te[c].astype(str).fillna("__MISSING__")

    for c in num_cols:
        tr[c] = pd.to_numeric(tr[c], errors="coerce")
        te[c] = pd.to_numeric(te[c], errors="coerce")

    return tr, te, feature_cols, cat_cols, num_cols


def make_time_based_match_split(df: pd.DataFrame, valid_ratio: float = 0.10):
    match_dates = (
        df.groupby("match_id", as_index=False)["date"]
        .min()
        .sort_values("date")
        .reset_index(drop=True)
    )

    split_idx = int(len(match_dates) * (1 - valid_ratio))
    train_match_ids = set(match_dates.loc[: split_idx - 1, "match_id"])
    valid_match_ids = set(match_dates.loc[split_idx:, "match_id"])

    train_mask = df["match_id"].isin(train_match_ids)
    valid_mask = df["match_id"].isin(valid_match_ids)

    return train_mask, valid_mask, match_dates


train_fe, test_fe, feature_cols, cat_cols, num_cols = build_feature_frame(train, test)
train_mask, valid_mask, match_dates = make_time_based_match_split(train_fe, valid_ratio=0.10)

print("Train rows   :", int(train_mask.sum()))
print("Valid rows   :", int(valid_mask.sum()))
print("Train matches:", train_fe.loc[train_mask, "match_id"].nunique())
print("Valid matches:", train_fe.loc[valid_mask, "match_id"].nunique())

assert set(train_fe.loc[train_mask, "match_id"]).isdisjoint(set(train_fe.loc[valid_mask, "match_id"]))

## 5. Official AW-MAE Evaluator

Evaluator di bawah ini mengikuti formula resmi:
- base MAE per pertandingan
- penalti exact / outcome / goal-difference
- outcome multiplier
- loss non-linear pangkat `1.5`
- weighted average berdasarkan tournament

Karena model dilatih row-wise, evaluator akan:
1. menerima prediksi row-wise,
2. membangun representasi canonical per `match_id`,
3. menghitung skor **sekali per pertandingan**.

In [ ]:
def get_tournament_weight(tournament: str) -> float:
    t = str(tournament).lower().strip()

    if "fifa world cup" in t or t == "world cup":
        return 2.00
    if "afc championship" in t or "afc asian cup" in t or "asian cup" in t:
        return 1.80
    if "friendly" in t:
        return 0.96
    return 1.20


def aggregate_match_predictions(df_rows, pred_team_cont, pred_opp_cont):
    tmp = df_rows[["match_id", "Id", "team", "opponent", "team_goals", "opp_goals", "tournament", "date"]].copy()
    tmp["pred_team_cont"] = np.asarray(pred_team_cont, dtype=float)
    tmp["pred_opp_cont"] = np.asarray(pred_opp_cont, dtype=float)

    rows = []
    for match_id, g in tmp.groupby("match_id", sort=False):
        g = g.sort_values("Id").reset_index(drop=True)
        r1 = g.iloc[0]

        if len(g) == 2:
            r2 = g.iloc[1]
            pred_a = float(np.mean([r1["pred_team_cont"], r2["pred_opp_cont"]]))
            pred_b = float(np.mean([r1["pred_opp_cont"], r2["pred_team_cont"]]))
        else:
            pred_a = float(r1["pred_team_cont"])
            pred_b = float(r1["pred_opp_cont"])

        rows.append(
            {
                "match_id": match_id,
                "canonical_id": r1["Id"],
                "team_a": r1["team"],
                "team_b": r1["opponent"],
                "team_goals_true": int(r1["team_goals"]),
                "opp_goals_true": int(r1["opp_goals"]),
                "pred_team_cont": pred_a,
                "pred_opp_cont": pred_b,
                "tournament": r1["tournament"],
                "date": r1["date"],
            }
        )
    return pd.DataFrame(rows)


def postprocess_match_predictions(match_pred_df, strategy="round", draw_shrink=0.0):
    out = match_pred_df.copy()
    a = out["pred_team_cont"].astype(float).values.copy()
    b = out["pred_opp_cont"].astype(float).values.copy()

    if draw_shrink > 0:
        mean_score = (a + b) / 2.0
        a = (1 - draw_shrink) * a + draw_shrink * mean_score
        b = (1 - draw_shrink) * b + draw_shrink * mean_score

    if strategy == "round":
        a = np.rint(a)
        b = np.rint(b)
    elif strategy == "floor":
        a = np.floor(a)
        b = np.floor(b)
    elif strategy == "ceil":
        a = np.ceil(a)
        b = np.ceil(b)
    elif strategy == "hybrid":
        a = np.where(a < 0.75, np.floor(a), np.rint(a))
        b = np.where(b < 0.75, np.floor(b), np.rint(b))
    else:
        raise ValueError(strategy)

    out["pred_team_goals"] = np.clip(a, 0, None).astype(int)
    out["pred_opp_goals"] = np.clip(b, 0, None).astype(int)
    return out


def evaluate_awmae_from_match_df(match_df):
    df = match_df.copy()

    yt = df["team_goals_true"].astype(int).values
    yo = df["opp_goals_true"].astype(int).values
    pt = df["pred_team_goals"].astype(int).values
    po = df["pred_opp_goals"].astype(int).values

    base_mae = (np.abs(yt - pt) + np.abs(yo - po)) / 2.0

    exact = ((yt == pt) & (yo == po)).astype(int)
    outcome_true = np.sign(yt - yo)
    outcome_pred = np.sign(pt - po)
    outcome_ok = (outcome_true == outcome_pred).astype(int)
    gd_ok = ((yt - yo) == (pt - po)).astype(int)

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome_ok) + 0.15 * (1 - gd_ok)
    multiplier = np.where(outcome_ok == 1, 1.0, 1.5)

    raw_loss = base_mae + penalty
    loss = np.power(raw_loss * multiplier, 1.5)

    weights = df["tournament"].apply(get_tournament_weight).astype(float).values
    awmae = float(np.sum(loss * weights) / np.sum(weights))

    return {
        "awmae": awmae,
        "base_mae": float(np.mean(base_mae)),
        "exact_rate": float(np.mean(exact)),
        "outcome_accuracy": float(np.mean(outcome_ok)),
        "gd_accuracy": float(np.mean(gd_ok)),
        "n_matches": int(len(df)),
    }


def evaluate_rowwise_predictions(df_rows, pred_team_cont, pred_opp_cont, strategy="round", draw_shrink=0.0):
    match_df = aggregate_match_predictions(df_rows, pred_team_cont, pred_opp_cont)
    match_df = postprocess_match_predictions(match_df, strategy=strategy, draw_shrink=draw_shrink)
    metrics = evaluate_awmae_from_match_df(match_df)
    return metrics, match_df

## 6. Modeling — Dual Regression

Baseline utama memakai `CatBoostRegressor` karena cocok untuk mixed tabular data dan bisa menangani fitur kategorikal dengan baik.

In [ ]:
from catboost import CatBoostRegressor

X_train = train_fe.loc[train_mask, feature_cols].copy()
X_valid = train_fe.loc[valid_mask, feature_cols].copy()

y_train_team = train_fe.loc[train_mask, "team_goals"].astype(float)
y_train_opp = train_fe.loc[train_mask, "opp_goals"].astype(float)
y_valid_team = train_fe.loc[valid_mask, "team_goals"].astype(float)
y_valid_opp = train_fe.loc[valid_mask, "opp_goals"].astype(float)

team_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5.0,
    iterations=400,
    random_seed=SEED,
    verbose=False,
)

opp_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5.0,
    iterations=400,
    random_seed=SEED,
    verbose=False,
)

team_model.fit(
    X_train,
    y_train_team,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid_team),
    use_best_model=True,
    verbose=False,
)

opp_model.fit(
    X_train,
    y_train_opp,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid_opp),
    use_best_model=True,
    verbose=False,
)

valid_pred_team_cont = team_model.predict(X_valid)
valid_pred_opp_cont = opp_model.predict(X_valid)

## 7. Post-processing

Output model regresi masih continuous, jadi perlu diubah menjadi skor integer valid.

Kita bandingkan beberapa strategi sederhana dan memilih yang terbaik berdasarkan **AW-MAE validation**.

In [ ]:
results = []

for strategy in ["round", "floor", "ceil", "hybrid"]:
    for draw_shrink in [0.00, 0.05, 0.10, 0.15]:
        metrics, _ = evaluate_rowwise_predictions(
            df_rows=train_fe.loc[valid_mask].copy(),
            pred_team_cont=valid_pred_team_cont,
            pred_opp_cont=valid_pred_opp_cont,
            strategy=strategy,
            draw_shrink=draw_shrink,
        )
        results.append({"strategy": strategy, "draw_shrink": draw_shrink, **metrics})

results_df = pd.DataFrame(results).sort_values(
    ["awmae", "base_mae", "outcome_accuracy"],
    ascending=[True, True, False]
).reset_index(drop=True)

display(results_df.head(12))
best_cfg = results_df.iloc[0].to_dict()
print("Best config:", best_cfg)

## 8. Evaluation

Section ini menampilkan:
- AW-MAE validation
- base MAE
- exact score rate
- outcome accuracy
- GD accuracy
- contoh tabel prediksi vs aktual di level pertandingan

In [ ]:
best_metrics, valid_match_pred = evaluate_rowwise_predictions(
    df_rows=train_fe.loc[valid_mask].copy(),
    pred_team_cont=valid_pred_team_cont,
    pred_opp_cont=valid_pred_opp_cont,
    strategy=best_cfg["strategy"],
    draw_shrink=float(best_cfg["draw_shrink"]),
)

display(pd.DataFrame([best_metrics]))

comparison_df = valid_match_pred[
    [
        "match_id", "canonical_id", "team_a", "team_b",
        "team_goals_true", "opp_goals_true",
        "pred_team_goals", "pred_opp_goals",
        "tournament", "date"
    ]
].copy()

comparison_df["is_exact"] = (
    (comparison_df["team_goals_true"] == comparison_df["pred_team_goals"]) &
    (comparison_df["opp_goals_true"] == comparison_df["pred_opp_goals"])
)

display(comparison_df.head(20))

## 9. Error Analysis Singkat

Analisis singkat untuk melihat:
- apakah prediksi outcome sering benar,
- apakah model cenderung underpredict atau overpredict total gol.

In [ ]:
error_df = valid_match_pred.copy()
error_df["true_total_goals"] = error_df["team_goals_true"] + error_df["opp_goals_true"]
error_df["pred_total_goals"] = error_df["pred_team_goals"] + error_df["pred_opp_goals"]
error_df["total_goal_error"] = error_df["pred_total_goals"] - error_df["true_total_goals"]
error_df["is_outcome_correct"] = (
    np.sign(error_df["team_goals_true"] - error_df["opp_goals_true"]) ==
    np.sign(error_df["pred_team_goals"] - error_df["pred_opp_goals"])
)

display(
    error_df[
        [
            "match_id", "team_a", "team_b",
            "team_goals_true", "opp_goals_true",
            "pred_team_goals", "pred_opp_goals",
            "true_total_goals", "pred_total_goals",
            "total_goal_error", "is_outcome_correct"
        ]
    ].head(20)
)

print("Mean total-goal error:", error_df["total_goal_error"].mean())
print("Outcome accuracy      :", error_df["is_outcome_correct"].mean())

## 10. Final Test Inference

Model dilatih ulang pada seluruh data train, lalu dipakai untuk prediksi `test.csv`.

In [ ]:
X_full = train_fe[feature_cols].copy()
y_full_team = train_fe["team_goals"].astype(float)
y_full_opp = train_fe["opp_goals"].astype(float)
X_test = test_fe[feature_cols].copy()

final_team_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5.0,
    iterations=400,
    random_seed=SEED,
    verbose=False,
)

final_opp_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5.0,
    iterations=400,
    random_seed=SEED,
    verbose=False,
)

final_team_model.fit(X_full, y_full_team, cat_features=cat_cols, verbose=False)
final_opp_model.fit(X_full, y_full_opp, cat_features=cat_cols, verbose=False)

test_pred_team_cont = final_team_model.predict(X_test)
test_pred_opp_cont = final_opp_model.predict(X_test)

## 11. Submission CSV

Aturan yang dipenuhi:
- CSV dibuat dari `test.csv`
- hanya ada kolom `Id`, `team_goals`, `opp_goals`
- jumlah baris sama dengan `test.csv`
- nilai `Id` berasal dari `test.csv`
- target final integer dan minimal 0

In [ ]:
def reconcile_test_predictions(test_df, pred_team_cont, pred_opp_cont, strategy, draw_shrink):
    tmp = test_df[["Id", "match_id"]].copy()
    tmp["pred_team_cont"] = np.asarray(pred_team_cont, dtype=float)
    tmp["pred_opp_cont"] = np.asarray(pred_opp_cont, dtype=float)

    rows = []
    for match_id, g in tmp.groupby("match_id", sort=False):
        g = g.sort_values("Id").reset_index(drop=True)
        r1 = g.iloc[0]

        if len(g) == 2:
            r2 = g.iloc[1]
            score_a = float(np.mean([r1["pred_team_cont"], r2["pred_opp_cont"]]))
            score_b = float(np.mean([r1["pred_opp_cont"], r2["pred_team_cont"]]))
        else:
            score_a = float(r1["pred_team_cont"])
            score_b = float(r1["pred_opp_cont"])

        mean_score = (score_a + score_b) / 2.0
        score_a = (1 - draw_shrink) * score_a + draw_shrink * mean_score
        score_b = (1 - draw_shrink) * score_b + draw_shrink * mean_score

        if len(g) == 2:
            rows.append({"Id": g.iloc[0]["Id"], "team_goals_cont": score_a, "opp_goals_cont": score_b})
            rows.append({"Id": g.iloc[1]["Id"], "team_goals_cont": score_b, "opp_goals_cont": score_a})
        else:
            rows.append({"Id": r1["Id"], "team_goals_cont": score_a, "opp_goals_cont": score_b})

    sub = pd.DataFrame(rows)

    if strategy == "round":
        sub["team_goals"] = np.rint(sub["team_goals_cont"])
        sub["opp_goals"] = np.rint(sub["opp_goals_cont"])
    elif strategy == "floor":
        sub["team_goals"] = np.floor(sub["team_goals_cont"])
        sub["opp_goals"] = np.floor(sub["opp_goals_cont"])
    elif strategy == "ceil":
        sub["team_goals"] = np.ceil(sub["team_goals_cont"])
        sub["opp_goals"] = np.ceil(sub["opp_goals_cont"])
    elif strategy == "hybrid":
        sub["team_goals"] = np.where(sub["team_goals_cont"] < 0.75, np.floor(sub["team_goals_cont"]), np.rint(sub["team_goals_cont"]))
        sub["opp_goals"] = np.where(sub["opp_goals_cont"] < 0.75, np.floor(sub["opp_goals_cont"]), np.rint(sub["opp_goals_cont"]))
    else:
        raise ValueError(strategy)

    sub["team_goals"] = np.clip(sub["team_goals"], 0, None).astype(int)
    sub["opp_goals"] = np.clip(sub["opp_goals"], 0, None).astype(int)

    final_submission = test_df[["Id"]].merge(sub[["Id", "team_goals", "opp_goals"]], on="Id", how="left")
    final_submission[["team_goals", "opp_goals"]] = final_submission[["team_goals", "opp_goals"]].fillna(0).astype(int)

    return final_submission


submission = reconcile_test_predictions(
    test_df=test_fe.copy(),
    pred_team_cont=test_pred_team_cont,
    pred_opp_cont=test_pred_opp_cont,
    strategy=best_cfg["strategy"],
    draw_shrink=float(best_cfg["draw_shrink"]),
)

assert list(submission.columns) == ["Id", "team_goals", "opp_goals"]
assert len(submission) == len(test)
assert submission["Id"].equals(test["Id"])
assert submission["team_goals"].min() >= 0
assert submission["opp_goals"].min() >= 0

submission.to_csv(FINAL_CSV_NAME, index=False)
display(submission.head())
print("Saved to:", FINAL_CSV_NAME)

## 12. Conclusion

Notebook ini sudah mencakup:
- training pipeline
- evaluasi validation dengan official AW-MAE
- prediksi validation
- tabel hasil prediksi vs aktual
- ringkasan fitur yang dipakai
- ringkasan strategi post-processing
- inference pada `test.csv`
- pembuatan CSV final submission